# 🏥 Hospital Multilingual RAG System
## Retrieval-Augmented Generation with Ollama · FAISS · SQLite
### Languages: English | සිංහල (Sinhala) | தமிழ் (Tamil)

---
**What this notebook teaches you, step by step:**
1. Why RAG (Retrieval-Augmented Generation) exists and how it works
2. Setting up a local LLM using Ollama (no internet, no GPU needed)
3. Creating a hospital database in SQLite with real-world style data
4. Building a FAISS vector store for semantic search
5. A multilingual query pipeline supporting English, Sinhala, and Tamil
6. Effective query matching techniques (reranking, thresholding)
7. End-to-end integration and incremental testing
8. Launching the Streamlit web app

> **Layman analogy:** Think of RAG like a brilliant librarian (LLM) who reads your question, runs to find the right books (FAISS search), then composes a precise answer using only those books — not guessing from memory.


---
## 📦 Section 1 — Install Dependencies

**What this does:**  
Installs all the Python libraries this project needs.  
Think of it like buying all the ingredients before cooking.

| Package | Purpose |
|---|---|
| `ollama` | Talks to the local Ollama LLM server |
| `faiss-cpu` | Lightning-fast vector similarity search (CPU edition) |
| `sentence-transformers` | Converts text → numbers (embeddings) |
| `langchain` | Helper utilities for building RAG pipelines |
| `streamlit` | Builds the web interface |
| `pandas` | Tables and data wrangling |


In [1]:
# ╔══════════════════════════════════════════════════════╗
# ║  INSTALL — run once, then restart kernel if needed  ║
# ╚══════════════════════════════════════════════════════╝

import subprocess, sys

packages = [
    "ollama",
    "faiss-cpu",
    "sentence-transformers",
    "langchain",
    "langchain-community",
    "langchain-ollama",
    "streamlit",
    "pandas",
    "numpy",
    "tqdm",
]

for pkg in packages:
    print(f"Installing {pkg}...", end=" ")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", pkg, "-q"],
        capture_output=True, text=True
    )
    print("✅" if result.returncode == 0 else f"❌ {result.stderr[:60]}")

print("\n🎉 All packages installed!")


Installing ollama... ✅
Installing faiss-cpu... ✅
Installing sentence-transformers... ✅
Installing langchain... ✅
Installing langchain-community... ✅
Installing langchain-ollama... ✅
Installing streamlit... ✅
Installing pandas... ✅
Installing numpy... ✅
Installing tqdm... ✅

🎉 All packages installed!


---
## 🤖 Section 2 — Local LLM with Ollama

**What is Ollama?**  
Ollama is a tool that lets you run AI language models entirely on your own computer.  
No internet connection needed. No data sent to any cloud.  
It's like having a private ChatGPT that lives on your laptop.

**Why no GPU?**  
Models like `llama3.2:1b` (1 billion parameters) run acceptably on a modern CPU.  
We tune settings (`num_predict`, `num_ctx`) to keep responses fast.

**Setup steps (do this in terminal before running this notebook):**
```bash
# 1. Install Ollama from https://ollama.com
# 2. Pull a small, fast model:
ollama pull llama3.2:1b      # ~800 MB, fastest
ollama pull llama3.2         # ~2 GB, smarter
ollama pull gemma2:2b        # ~1.6 GB, Google's model
# 3. Start the server:
ollama serve
```


In [1]:
# ════════════════════════════════════════════════════════════
# TEST OLLAMA CONNECTION
# ════════════════════════════════════════════════════════════
# This cell pings the local Ollama server to make sure it's running.
# If you get a connection error, open a terminal and run: ollama serve

import ollama
import time

def test_ollama(model: str = "llama3.2:1b") -> bool:
    """
    Send a tiny test prompt to Ollama.
    Returns True if the server responds, False otherwise.
    """
    try:
        t0 = time.time()
        resp = ollama.chat(
            model=model,
            messages=[{"role": "user", "content": "Say: OK"}],
            options={"num_predict": 5, "temperature": 0},
        )
        elapsed = time.time() - t0
        print(f"✅ Ollama is running!  Model: {model}")
        print(f"   Response: {resp['message']['content'].strip()}")
        print(f"   Time    : {elapsed:.2f}s")
        return True
    except Exception as e:
        print(f"❌ Ollama not reachable: {e}")
        print("   → Run  `ollama serve`  in a terminal, then retry.")
        return False
        

# Choose the model you pulled
MODEL = "llama3.2:1b"   # change to "llama3.2" for a smarter but slower model
OLLAMA_OK = test_ollama(MODEL)


✅ Ollama is running!  Model: llama3.2:1b
   Response: OK.
   Time    : 20.18s


---
## 🗄️ Section 3 — Hospital Database (SQLite)

**What is SQLite?**  
SQLite is a file-based database — the entire database lives in a single `.db` file.  
No server installation needed. Perfect for demos and small apps.

**What data do we store?**
- `doctors` — specialist profiles, availability, multilingual bios
- `departments` — floor, phone, opening hours
- `services` — FAQs about appointments, emergency, labs, visiting hours

**Multilingual strategy:**  
We store bios/descriptions in English, Sinhala, and Tamil directly in the DB.  
The embedding model we use later understands all three scripts natively.


In [3]:
# ════════════════════════════════════════════════════════════
# BUILD THE HOSPITAL SQLite DATABASE
# ════════════════════════════════════════════════════════════
# We create 3 tables and seed them with realistic dummy data.
# This is the "knowledge base" our AI will draw answers from.

import sqlite3
import pandas as pd
from pathlib import Path

DB_PATH = Path("hospital.db")

def create_database(db_path: Path) -> sqlite3.Connection:
    conn = sqlite3.connect(str(db_path))
    cur  = conn.cursor()

    # ── DOCTORS ──────────────────────────────────────────────
    cur.execute("""
        CREATE TABLE IF NOT EXISTS doctors (
            id    INTEGER PRIMARY KEY,
            name  TEXT NOT NULL,
            dept  TEXT NOT NULL,
            lang  TEXT NOT NULL,
            bio   TEXT NOT NULL
        )
    """)

    doctor_data = [
        # English bios
        (1, "Dr. Amara Perera", "Cardiology", "en",
         "Dr. Amara Perera is a senior cardiologist with 20 years of experience. "
         "She specialises in interventional cardiology and heart failure management. "
         "Clinic: Monday, Wednesday, Friday 9am-1pm. Phone: 0112-345-001."),

        (2, "Dr. Suresh Kumar", "Neurology", "en",
         "Dr. Suresh Kumar leads the Neurology department. "
         "He treats epilepsy, stroke, Parkinson's disease, and migraines. "
         "Clinic hours: Tuesday and Thursday 10am-2pm. Phone: 0112-345-002."),

        (3, "Dr. Nimali Fernando", "Paediatrics", "en",
         "Dr. Nimali Fernando is a paediatrician caring for children from birth to 18. "
         "Vaccines, growth monitoring, asthma, and childhood illnesses. "
         "Appointments: weekdays 8am-12pm. Floor 2."),

        (4, "Dr. Ravi Jayasinghe", "Orthopaedics", "en",
         "Dr. Ravi Jayasinghe specialises in joint replacement and sports injuries. "
         "Knee replacement, hip replacement, shoulder surgery. "
         "Available: Wednesday and Saturday 9am-3pm. Floor 4."),

        (5, "Dr. Priya Balasingham", "Gynaecology", "en",
         "Dr. Priya Balasingham provides maternal and women's health services. "
         "Antenatal care, family planning, menopause management. "
         "Mon-Fri 9am-5pm. Floor 2."),

        (6, "Dr. Chamara Wijeratne", "Oncology", "en",
         "Dr. Chamara Wijejeratne is the head of Oncology. "
         "He treats breast cancer, lung cancer, and haematological malignancies. "
         "Tuesdays and Thursdays 9am-1pm. Floor 5."),

        (7, "Dr. Lakmini Dias", "Endocrinology", "en",
         "Dr. Lakmini Dias manages diabetes, thyroid disorders, and hormonal conditions. "
         "Clinic: Mondays and Fridays 10am-2pm. Floor 3."),

        # Sinhala bio
        (8, "Dr. Kamala Dissanayake", "General Medicine", "si",
         "Dr. Kamala Dissanayake සාමාන්‍ය වෛද්‍ය විශේෂඥ. "
         "ඇය දියවැඩියාව, රුධිර පීඩනය, උණ සහ සාමාන්‍ය රෝග සඳහා ප්‍රතිකාර කරයි. "
         "වේලාව: සඳුදා සිට සිකුරාදා 8am-4pm. 1 වන මහල."),

        # Tamil bio
        (9, "Dr. Vijay Arulrajah", "Dermatology", "ta",
         "Dr. Vijay Arulrajah தோல் நோய் நிபுணர். "
         "சொரியாஸிஸ், அரிப்பு, முகப்பரு மற்றும் தோல் புற்றுநோய் சிகிச்சை வழங்குகிறார். "
         "நேரம்: திங்கள், புதன், வெள்ளி 9am-1pm. இரண்டாவது மாடி."),

        (10, "Dr. Ananthi Sivakumar", "Cardiology", "ta",
         "Dr. Ananthi Sivakumar இதய நோய் நிபுணர். "
         "மாரடைப்பு, இதய செயலிழப்பு மற்றும் உயர் இரத்த அழுத்தம் ஆகியவற்றை சிகிச்சை செய்கிறார். "
         "கிளினிக்: செவ்வாய், வியாழன் 9am-1pm."),
    ]

    cur.executemany(
        "INSERT OR IGNORE INTO doctors VALUES (?,?,?,?,?)",
        doctor_data
    )

    # ── DEPARTMENTS ───────────────────────────────────────────
    cur.execute("""
        CREATE TABLE IF NOT EXISTS departments (
            id    INTEGER PRIMARY KEY,
            name  TEXT NOT NULL,
            floor INTEGER NOT NULL,
            phone TEXT,
            hours TEXT
        )
    """)

    dept_data = [
        (1,  "Cardiology",       3, "0112-345-001", "Mon-Fri 8am-5pm"),
        (2,  "Neurology",        3, "0112-345-002", "Mon-Fri 9am-4pm"),
        (3,  "Paediatrics",      2, "0112-345-003", "Mon-Sat 8am-6pm"),
        (4,  "Orthopaedics",     4, "0112-345-004", "Mon-Fri 8am-5pm"),
        (5,  "Gynaecology",      2, "0112-345-005", "Mon-Fri 9am-5pm"),
        (6,  "Oncology",         5, "0112-345-006", "Tue-Thu 9am-1pm"),
        (7,  "Endocrinology",    3, "0112-345-007", "Mon, Fri 10am-2pm"),
        (8,  "General Medicine", 1, "0112-345-008", "Mon-Sat 7am-7pm"),
        (9,  "Dermatology",      2, "0112-345-009", "Mon-Fri 9am-4pm"),
        (10, "Emergency",        0, "0112-345-911", "24 hours / 7 days"),
        (11, "Pharmacy",         1, "0112-345-010", "Mon-Sun 7am-10pm"),
        (12, "Radiology",        1, "0112-345-011", "Mon-Fri 8am-8pm"),
        (13, "Pathology Lab",    1, "0112-345-012", "Mon-Sat 6am-6pm"),
        (14, "ICU",              4, "0112-345-013", "24 hours — no direct calls"),
    ]

    cur.executemany(
        "INSERT OR IGNORE INTO departments VALUES (?,?,?,?,?)",
        dept_data
    )

    # ── SERVICES / FAQ ────────────────────────────────────────
    cur.execute("""
        CREATE TABLE IF NOT EXISTS services (
            id     INTEGER PRIMARY KEY,
            title  TEXT NOT NULL,
            lang   TEXT NOT NULL,
            detail TEXT NOT NULL
        )
    """)

    service_data = [
        (1,  "Appointment Booking", "en",
         "To book an appointment call 0112-345-000 or visit reception on Floor 1. "
         "Online booking: hospital.lk/appointments. Bring National ID. "
         "Cancellations must be made 24 hours in advance."),

        (2,  "Emergency Services", "en",
         "Our 24-hour Emergency Department is on the Ground Floor. "
         "Ambulance: 1990. The ER handles trauma, chest pain, stroke, "
         "severe breathing, unconsciousness, and all life-threatening conditions. "
         "Triage nurses assess severity within 5 minutes of arrival."),

        (3,  "Lab Tests", "en",
         "Pathology Lab is on Floor 1, open Mon-Sat 6am-6pm. "
         "Fasting blood tests must be done before 10am. "
         "Results online within 24 hours via patient portal at hospital.lk/results. "
         "Urine, stool, and culture tests available same day."),

        (4,  "Visiting Hours", "en",
         "General wards: 4pm-6pm weekdays; 10am-12pm and 4pm-6pm weekends. "
         "ICU: 10am-11am and 4pm-5pm daily, max 2 visitors per patient. "
         "Children under 12 not permitted in ICU or surgical wards."),

        (5,  "Insurance & Billing", "en",
         "We accept all major Sri Lankan health insurance. "
         "Cashless: Ceylinco, AIA, Softlogic, Union Assurance. "
         "Billing office: Floor 1, Mon-Fri 8am-5pm. Phone: 0112-345-099."),

        (6,  "Pharmacy", "en",
         "Hospital pharmacy is on Floor 1, open Mon-Sun 7am-10pm. "
         "Prescriptions from our doctors are filled within 20 minutes. "
         "Generic medicines available at reduced cost. Phone: 0112-345-010."),

        (7,  "Radiology & Imaging", "en",
         "X-ray, CT scan, MRI, and ultrasound available on Floor 1. "
         "CT and MRI require a doctor's referral. "
         "X-ray results in 1 hour. MRI reports in 24 hours. Phone: 0112-345-011."),

        (8,  "Parking", "en",
         "Free parking for up to 2 hours for patients and visitors. "
         "Enter from Hospital Road South Gate. "
         "Paid parking thereafter: Rs. 100/hour. Disability bays at main entrance."),

        (9,  "Patient Portal", "en",
         "Register at hospital.lk/portal to view test results, book appointments, "
         "and download discharge summaries. "
         "Requires a valid email and mobile number used at registration."),

        # ── Sinhala services ─────────────────────────────────
        (10, "හදිසි සේවා", "si",
         "හදිසි ගිලන් රථ අංකය: 1990. "
         "හෘදයාබාධ, ආඝාත රෝග ලක්ෂණ, දැඩි ලේ ගැලීම ඇතිවිට "
         "වහාම ගිලන් රථ ඇමතිය යුතුය. "
         "හදිසි ඒකකය 24 පැය, සතිය පුරා විවෘතව ඇත. ශූන්‍ය මහල."),

        (11, "හමුවීම් (Appointment) ලබා ගැනීම", "si",
         "හමුවීමක් ලබා ගැනීම සඳහා 0112-345-000 අංකය අමතන්න "
         "හෝ 1 වන මහලේ 접수 (reception) ට ගොඩ ගන්න. "
         "ජාතික හැඳුනුම්පත රැගෙන එන්න."),

        (12, "ළමා රෝග ශාඛාව", "si",
         "ළමා රෝග ශාඛාව 2 වන මහලෙහි ඇත. "
         "Dr. Nimali Fernando ළමා රෝග විශේෂඥ. "
         "සඳුදා සිට සිකුරාදා 8am-12pm. "
         "ළමුන්ගේ එන්නත් සහ වර්ධනය නිරීක්ෂණ සේවාව ලබාදෙයි."),

        (13, "රසායනාගාර පරීක්ෂණ", "si",
         "රසායනාගාරය 1 වන මහලෙහි ඇත. "
         "සඳුදා සිට සෙනසුරාදා 6am-6pm දක්වා විවෘතය. "
         "නිරාහාර (fasting) රුධිර පරීක්ෂණ 10am ට පෙර ගත යුතුය. "
         "ප්‍රතිඵල: hospital.lk/results."),

        # ── Tamil services ────────────────────────────────────
        (14, "அவசர சேவைகள்", "ta",
         "அவசர சேவை எண்: 1990. "
         "மாரடைப்பு, பக்கவாதம், அதிரடி காயம், உணர்விழப்பு ஆகியவற்றிற்கு "
         "உடனடியாக அவசர ஆம்புலன்ஸை அழைக்கவும். "
         "அவசர பிரிவு 24 மணி நேரமும், 7 நாட்களும் திறந்திருக்கும். தரை மாடி."),

        (15, "சந்திப்பு பதிவு", "ta",
         "சந்திப்பு பதிவு செய்ய 0112-345-000 என்ற எண்ணில் அழைக்கவும் "
         "அல்லது முதல் மாடியில் உள்ள வரவேற்பு மையத்தை சந்திக்கவும். "
         "தேசிய அடையாள அட்டை கொண்டு வரவும்."),

        (16, "மருந்தகம் (Pharmacy)", "ta",
         "மருந்தகம் முதல் மாடியில் உள்ளது. "
         "திங்கள் முதல் ஞாயிறு வரை 7am-10pm திறந்திருக்கும். "
         "மருத்துவர் பரிந்துரை மருந்துகள் 20 நிமிடத்தில் வழங்கப்படும்."),

        (17, "ஆய்வக பரிசோதனைகள்", "ta",
         "பரிசோதனை ஆய்வகம் முதல் மாடியில் உள்ளது. "
         "திங்கள் முதல் சனிக்கிழமை 6am-6pm திறந்திருக்கும். "
         "உண்ணாவிரத (fasting) இரத்த பரிசோதனைகள் காலை 10 மணிக்கு முன் செய்யப்பட வேண்டும்."),
    ]

    cur.executemany(
        "INSERT OR IGNORE INTO services VALUES (?,?,?,?)",
        service_data
    )

    conn.commit()
    print(f"✅ Database created at: {db_path.resolve()}")
    return conn

# Create and open the database
conn = create_database(DB_PATH)

# ── Quick inspection ──────────────────────────────────────────────
print("\n📊 Database summary:")
cur = conn.cursor()
for tbl in ["doctors", "departments", "services"]:
    n = cur.execute(f"SELECT COUNT(*) FROM {tbl}").fetchone()[0]
    print(f"   {tbl:20s}: {n} rows")


✅ Database created at: D:\Hospital_RAG_System\hospital.db

📊 Database summary:
   doctors             : 10 rows
   departments         : 14 rows
   services            : 17 rows


In [4]:
# ════════════════════════════════════════════════════════════
# PREVIEW THE DATA AS NICE TABLES
# ════════════════════════════════════════════════════════════
# Pandas lets us display the SQLite rows in a readable format
# right here in the notebook.

import pandas as pd

print("=== DOCTORS ===")
display(pd.read_sql("SELECT id, name, dept, lang FROM doctors", conn))

print("\n=== DEPARTMENTS ===")
display(pd.read_sql("SELECT * FROM departments", conn))

print("\n=== SERVICES (first 5) ===")
display(pd.read_sql("SELECT id, title, lang, detail FROM services LIMIT 5", conn))


=== DOCTORS ===


,id,name,dept,lang
0,1,Dr. Amara Perera,Cardiology,en
1,2,Dr. Suresh Kumar,Neurology,en
2,3,Dr. Nimali Fernando,Paediatrics,en
3,4,Dr. Ravi Jayasinghe,Orthopaedics,en
4,5,Dr. Priya Balasingham,Gynaecology,en
5,6,Dr. Chamara Wijeratne,Oncology,en
6,7,Dr. Lakmini Dias,Endocrinology,en
7,8,Dr. Kamala Dissanayake,General Medicine,si
8,9,Dr. Vijay Arulrajah,Dermatology,ta
9,10,Dr. Ananthi Sivakumar,Cardiology,ta



=== DEPARTMENTS ===


,id,name,floor,phone,hours
0,1,Cardiology,3,0112-345-001,Mon-Fri 8am-5pm
1,2,Neurology,3,0112-345-002,Mon-Fri 9am-4pm
2,3,Paediatrics,2,0112-345-003,Mon-Sat 8am-6pm
3,4,Orthopaedics,4,0112-345-004,Mon-Fri 8am-5pm
4,5,Gynaecology,2,0112-345-005,Mon-Fri 9am-5pm
5,6,Oncology,5,0112-345-006,Tue-Thu 9am-1pm
6,7,Endocrinology,3,0112-345-007,"Mon, Fri 10am-2pm"
7,8,General Medicine,1,0112-345-008,Mon-Sat 7am-7pm
8,9,Dermatology,2,0112-345-009,Mon-Fri 9am-4pm
9,10,Emergency,0,0112-345-911,24 hours / 7 days



=== SERVICES (first 5) ===


,id,title,lang,detail
0,1,Appointment Booking,en,To book an appointment call 0112-345-000 or vi...
1,2,Emergency Services,en,Our 24-hour Emergency Department is on the Gro...
2,3,Lab Tests,en,"Pathology Lab is on Floor 1, open Mon-Sat 6am-..."
3,4,Visiting Hours,en,General wards: 4pm-6pm weekdays; 10am-12pm and...
4,5,Insurance & Billing,en,We accept all major Sri Lankan health insuranc...


---
## 🔢 Section 4 — Multilingual Text Embeddings

**What is an embedding?**  
An embedding converts a sentence into a list of numbers (a vector).  
Sentences with similar meaning end up with similar numbers — even across languages!

**Layman analogy:**  
Imagine turning each sentence into a GPS coordinate.  
"Where is the cardiologist?" and "ஹார்ட் டாக்டர் யார்?" might both land  
near coordinate (0.72, -0.31, 0.44, …) because they mean the same thing.  
FAISS then finds the closest coordinates to your query.

**Model chosen:** `paraphrase-multilingual-MiniLM-L12-v2`  
- Supports 50+ languages including Sinhala and Tamil  
- Only ~120 MB — loads quickly on CPU  
- 384-dimension vectors — compact yet expressive


In [ ]:
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"

from sentence_transformers import SentenceTransformer
model = SentenceTransformer("intfloat/multilingual-e5-base")

D:\Hospital_RAG_System\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# ════════════════════════════════════════════════════════════
# LOAD THE MULTILINGUAL EMBEDDING MODEL
# ════════════════════════════════════════════════════════════
# SentenceTransformer downloads the model on first run (~120 MB).
# Subsequent runs load from cache (instant).

from sentence_transformers import SentenceTransformer
import numpy as np
import time

print("Loading multilingual embedding model…")
t0 = time.time()

EMBED_MODEL_NAME = "intfloat/multilingual-e5-base"
embed_model = SentenceTransformer(EMBED_MODEL_NAME, device="cpu")

print(f"✅ Model loaded in {time.time()-t0:.1f}s")
print(f"   Embedding dimension: {embed_model.get_sentence_embedding_dimension()}")


In [ ]:
# ════════════════════════════════════════════════════════════
# DEMONSTRATE CROSS-LINGUAL SEMANTIC SIMILARITY
# ════════════════════════════════════════════════════════════
# This cell proves the model understands that the same question
# asked in English, Sinhala, and Tamil should have similar embeddings.

from sklearn.metrics.pairwise import cosine_similarity

test_sentences = [
    # English
    "Where is the emergency department?",
    "What is the ambulance number?",
    # Sinhala
    "හදිසි ඒකකය කොතනද?",          # Where is the emergency unit?
    # Tamil
    "அவசர பிரிவு எங்கே உள்ளது?",   # Where is the emergency department?
    # Unrelated
    "The weather is sunny today.",
]

vecs = embed_model.encode(test_sentences, normalize_embeddings=True)
sim_matrix = cosine_similarity(vecs)

print("Cosine Similarity Matrix (1.0 = identical meaning):\n")
labels = ["EN:emergency","EN:ambulance","SI:emergency","TA:emergency","EN:weather"]
df_sim = pd.DataFrame(sim_matrix.round(3), index=labels, columns=labels)
display(df_sim)

print("\n✅ Notice: EN/SI/TA emergency questions score HIGH similarity (~0.7+)")
print("   'Weather' scores LOW similarity — the model understands context!")


---
## ⚡ Section 5 — FAISS Vector Index

**What is FAISS?**  
FAISS (Facebook AI Similarity Search) is a library for finding the most  
similar items in a large list of number-vectors — blazing fast.

**Layman analogy:**  
You have 10,000 library books. Each is turned into GPS coordinates.  
FAISS instantly finds the 5 books whose coordinates are closest to your query's coordinates.

**Index type we use:** `IndexFlatIP` (Inner Product)  
When vectors are normalised (length = 1), inner product = cosine similarity.  
This is the most accurate FAISS index — no approximation, no GPU needed.

**CPU optimisation tips applied:**
- Batch encoding with `batch_size=16` — reduces peak RAM
- `normalize_embeddings=True` — enables cosine via dot-product (faster than explicit normalisation)
- Flat index is fast for <50k documents — perfect for a hospital KB


In [4]:
# ════════════════════════════════════════════════════════════
# EXTRACT ALL KNOWLEDGE FROM SQLite INTO TEXT CHUNKS
# ════════════════════════════════════════════════════════════
# Each database row becomes one "chunk" — a text snippet
# the AI can retrieve and quote from.

def load_chunks_from_db(conn: sqlite3.Connection) -> list[dict]:
    """
    Pull every row from doctors, departments, services.
    Return a list of dicts with 'id', 'source', and 'text'.
    """
    chunks = []
    cur = conn.cursor()

    # ── Doctors ──────────────────────────────────────────────
    for doc_id, name, dept, lang, bio in cur.execute(
            "SELECT id, name, dept, lang, bio FROM doctors"):
        chunks.append({
            "id"     : f"doctor_{doc_id}",
            "source" : "doctors",
            "lang"   : lang,
            "text"   : f"Doctor Profile — {name} ({dept}): {bio}",
        })

    # ── Departments ───────────────────────────────────────────
    for d_id, name, floor, phone, hours in cur.execute(
            "SELECT id, name, floor, phone, hours FROM departments"):
        chunks.append({
            "id"     : f"dept_{d_id}",
            "source" : "departments",
            "lang"   : "en",
            "text"   : (
                f"Department: {name}. "
                f"Location: Floor {floor}. "
                f"Phone: {phone}. "
                f"Opening hours: {hours}."
            ),
        })

    # ── Services / FAQ ────────────────────────────────────────
    for s_id, title, lang, detail in cur.execute(
            "SELECT id, title, lang, detail FROM services"):
        chunks.append({
            "id"     : f"svc_{s_id}",
            "source" : "services",
            "lang"   : lang,
            "text"   : f"{title}: {detail}",
        })

    return chunks

ALL_CHUNKS = load_chunks_from_db(conn)
print(f"✅ Loaded {len(ALL_CHUNKS)} text chunks from the database")
print(f"   Sample chunk:\n   {ALL_CHUNKS[0]['text'][:200]}…")


NameError: name 'sqlite3' is not defined

In [2]:
# ════════════════════════════════════════════════════════════
# EMBED ALL CHUNKS AND BUILD THE FAISS INDEX
# ════════════════════════════════════════════════════════════
# We convert every text chunk into a vector and load them
# into FAISS so we can search by similarity later.

import faiss
from tqdm.auto import tqdm

def build_faiss_index(chunks: list[dict], model: SentenceTransformer):
    """
    1. Extract text from every chunk
    2. Encode in batches (CPU-friendly)
    3. Build a FAISS IndexFlatIP (cosine similarity on normalised vectors)
    Return: (faiss_index, ordered list of chunks matching index positions)
    """
    texts = [c["text"] for c in chunks]

    print(f"Embedding {len(texts)} chunks…")
    t0 = time.time()

    # batch_size=16 → lower peak RAM; show_progress_bar gives a live ticker
    vectors = model.encode(
        texts,
        batch_size=16,
        show_progress_bar=True,
        normalize_embeddings=True,   # critical: makes dot-product = cosine
    ).astype("float32")

    print(f"✅ Embedded in {time.time()-t0:.1f}s | shape: {vectors.shape}")

    dim = vectors.shape[1]
    index = faiss.IndexFlatIP(dim)   # Inner Product on unit vectors = cosine
    index.add(vectors)

    print(f"   FAISS index size: {index.ntotal} vectors")
    return index, chunks   # chunks list = our lookup table by FAISS rank

faiss_index, chunk_map = build_faiss_index(ALL_CHUNKS, embed_model)
print(f"\n✅ FAISS ready! Will search {faiss_index.ntotal} vectors in milliseconds.")


ImportError: DLL load failed while importing _swigfaiss: The specified module could not be found.

In [ ]:
# ════════════════════════════════════════════════════════════
# SAVE & RELOAD FAISS INDEX (optional — speeds up restarts)
# ════════════════════════════════════════════════════════════
# Saving means you don't have to re-embed on every notebook restart.

import json

FAISS_INDEX_PATH = "hospital_faiss.index"
CHUNKS_JSON_PATH = "hospital_chunks.json"

# Save
faiss.write_index(faiss_index, FAISS_INDEX_PATH)
with open(CHUNKS_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(chunk_map, f, ensure_ascii=False, indent=2)

print(f"✅ Saved FAISS index  → {FAISS_INDEX_PATH}")
print(f"   Saved chunk map    → {CHUNKS_JSON_PATH}")

# Reload (to verify)
faiss_index_loaded = faiss.read_index(FAISS_INDEX_PATH)
with open(CHUNKS_JSON_PATH, encoding="utf-8") as f:
    chunk_map_loaded = json.load(f)

print(f"\n✅ Reload successful! Index has {faiss_index_loaded.ntotal} vectors.")


---
## 🔍 Section 6 — Query Matching & Retrieval

**What is RAG retrieval?**  
When a user asks a question:
1. We embed the question → get its vector
2. We search FAISS → find the top-K most similar chunks
3. We filter by a relevance threshold → remove low-quality matches
4. We pass the surviving chunks to the LLM as context

**Optimisations included:**
- **Score threshold (0.25):** Chunks below this similarity are dropped — prevents hallucination from irrelevant context
- **Language-aware re-ranking:** Chunks matching the user's language script get a small bonus score
- **Query expansion:** We also search for the English paraphrase of Sinhala/Tamil queries to catch English-only content


In [ ]:
# ════════════════════════════════════════════════════════════
# LANGUAGE DETECTION UTILITY
# ════════════════════════════════════════════════════════════
# Detects script from Unicode code points — no ML needed,
# instant execution even on the slowest CPU.

def detect_language(text: str) -> str:
    """
    Detect language by Unicode script range.
    Sinhala: U+0D80–U+0DFF
    Tamil  : U+0B80–U+0BFF
    Default: English
    """
    for ch in text:
        cp = ord(ch)
        if 0x0D80 <= cp <= 0x0DFF:
            return "si"   # Sinhala
        if 0x0B80 <= cp <= 0x0BFF:
            return "ta"   # Tamil
    return "en"

# Test
samples = [
    "Where is the cardiologist?",
    "හදිසි ඒකකය කොතනද?",
    "அவசர சேவை எண் என்ன?",
]
for s in samples:
    print(f"  '{s[:40]}' → {detect_language(s)}")


In [ ]:
# ════════════════════════════════════════════════════════════
# RETRIEVAL FUNCTION WITH SMART FILTERING
# ════════════════════════════════════════════════════════════
# This is the heart of the RAG pipeline's retrieval stage.

def retrieve_chunks(
    query: str,
    faiss_idx,
    chunks: list[dict],
    embed_mdl: SentenceTransformer,
    top_k: int  = 6,
    threshold   : float = 0.25,
    lang_boost  : float = 0.05,
) -> list[dict]:
    """
    Retrieve the most relevant knowledge chunks for a query.

    Parameters
    ----------
    query      : User's question (any language)
    top_k      : How many candidates to retrieve from FAISS before filtering
    threshold  : Minimum cosine similarity — lower = noisy, higher = strict
    lang_boost : Extra score bonus for chunks in the same language as query

    Returns a ranked list of chunk dicts with an added 'score' key.
    """
    user_lang = detect_language(query)

    # 1️⃣  Embed the query (same model as chunks — critical!)
    q_vec = embed_mdl.encode(
        [query],
        normalize_embeddings=True,
    ).astype("float32")

    # 2️⃣  FAISS nearest-neighbour search
    scores, indices = faiss_idx.search(q_vec, top_k)

    # 3️⃣  Build result list with filtering and language boost
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx < 0 or idx >= len(chunks):
            continue
        chunk = chunks[idx].copy()
        adjusted = float(score)

        # Language-matching bonus (reward native-language content)
        if chunk.get("lang") == user_lang:
            adjusted += lang_boost

        if adjusted >= threshold:
            chunk["score"] = adjusted
            results.append(chunk)

    # 4️⃣  Sort by final score (descending)
    results.sort(key=lambda x: x["score"], reverse=True)
    return results

# ── Demo retrieval ─────────────────────────────────────────────
test_queries = [
    "Who is the heart doctor?",
    "හදිසි ගිලන් රථ අංකය කුමක්ද?",   # What is the emergency number?
    "அவசர சேவை எண் என்ன?",            # What is the emergency number?
    "What time does the pharmacy open?",
    "How do I book an appointment?",
]

for q in test_queries:
    hits = retrieve_chunks(q, faiss_index, chunk_map, embed_model, top_k=4)
    lang = detect_language(q)
    print(f"\n🔍 Query [{lang}]: {q[:55]}")
    for h in hits[:2]:
        print(f"   [{h['score']:.3f}] {h['text'][:90]}…")


---
## 🧠 Section 7 — Prompt Engineering for Multilingual RAG

**What is prompt engineering?**  
A "prompt" is the full instruction we send to the LLM.  
Good prompts = accurate, focused answers.  
Bad prompts = hallucinations and off-topic responses.

**Our prompt template includes:**
1. **Role instruction** — "You are a hospital assistant"
2. **Language instruction** — "Answer in Sinhala" (if query was Sinhala)
3. **Context block** — the retrieved chunks (what the AI can use)
4. **Strict grounding rule** — "Use ONLY the context below"
5. **The question** itself

**CPU speed tip:**  
We set `num_predict=512` to cap token generation. Shorter outputs = faster responses.


In [ ]:
# ════════════════════════════════════════════════════════════
# PROMPT TEMPLATES PER LANGUAGE
# ════════════════════════════════════════════════════════════

LANG_INSTRUCTIONS = {
    "en": (
        "Answer clearly and concisely in English. "
        "Use bullet points when listing multiple items."
    ),
    "si": (
        "පිළිතුර සිංහලෙන් පමණක් ලබා දෙන්න."
        "වෛද්‍ය වාක්‍ය (medical terms) ඉංග්‍රීසියෙන් ද ඇතුළත් කරන්න."
        "කෙටි හා පැහැදිලිව ලිවිය යුතුය."
        # Translation: Answer in Sinhala. Include medical terms in English too.
    ),
    "ta": (
        "தமிழில் பதில் அளிக்கவும். "
        "மருத்துவ சொற்கள் (medical terms) ஆங்கிலத்திலும் சேர்த்து வரவும். "
        "சுருக்கமாகவும் தெளிவாகவும் எழுதவும்."
        # Translation: Answer in Tamil. Include medical terms in English.
    ),
}

SYSTEM_TEMPLATE = """You are a helpful and accurate hospital information assistant.
{lang_instruction}

IMPORTANT: Use ONLY the information in the CONTEXT section below to answer.
If the answer is not in the context, say: "I don't have that information. 
Please contact reception at 0112-345-000."

Do NOT make up information. Do NOT guess.

--- CONTEXT ---
{context}
--- END CONTEXT ---

Question: {question}

Answer:"""

def build_prompt(query: str, chunks: list[dict]) -> str:
    """
    Assemble the final LLM prompt from the query and retrieved chunks.
    """
    lang = detect_language(query)
    lang_instr = LANG_INSTRUCTIONS.get(lang, LANG_INSTRUCTIONS["en"])

    # Format context: numbered chunks with source label
    context_lines = []
    for i, chunk in enumerate(chunks, 1):
        context_lines.append(
            f"[{i}] Source: {chunk['source']} | Score: {chunk['score']:.2f}\n"
            f"    {chunk['text']}"
        )
    context_block = "\n\n".join(context_lines)

    return SYSTEM_TEMPLATE.format(
        lang_instruction=lang_instr,
        context=context_block,
        question=query,
    )

# Demo prompt preview
demo_query = "What time does the pharmacy close?"
demo_chunks = retrieve_chunks(demo_query, faiss_index, chunk_map, embed_model, top_k=3)
demo_prompt = build_prompt(demo_query, demo_chunks)

print("📄 SAMPLE PROMPT:")
print("─" * 60)
print(demo_prompt[:800] + ("…" if len(demo_prompt) > 800 else ""))
print("─" * 60)
print(f"\nTotal prompt length: {len(demo_prompt)} characters")


---
## 🔗 Section 8 — End-to-End RAG Pipeline

**Full pipeline in one function:**  
`user question` → embed → FAISS search → filter → build prompt → Ollama LLM → answer

This section ties everything together and lets you test the full system  
before launching the Streamlit web interface.


In [ ]:
# ════════════════════════════════════════════════════════════
# OLLAMA LLM CALL — CPU OPTIMISED SETTINGS
# ════════════════════════════════════════════════════════════

def ask_llm(prompt: str, model: str = MODEL) -> tuple[str, float]:
    """
    Send a prompt to the local Ollama LLM and return (answer, seconds).

    CPU speed optimisations:
    - temperature=0.2 : deterministic → fewer tokens wasted on exploration
    - num_predict=512  : cap output length (shorter = faster)
    - num_ctx=2048     : smaller context window (faster attention)
    """
    if not OLLAMA_OK:
        return "[Ollama not available. Please run `ollama serve`]", 0.0

    t0 = time.time()
    try:
        resp = ollama.chat(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            options={
                "temperature" : 0.1,
                "num_predict" : 512,
                "num_ctx"     : 2048,
                "num_thread"  : 8,    # use 4 CPU threads
            },
        )
        return resp["message"]["content"].strip(), time.time() - t0
    except Exception as e:
        return f"Error: {e}", 0.0


def rag_answer(
    query: str,
    top_k     : int   = 5,
    threshold : float = 0.25,
    verbose   : bool  = True,
) -> dict:
    """
    Complete RAG pipeline:
    query → embed → retrieve → prompt → LLM → answer

    Returns a dict with all intermediate info for debugging.
    """
    lang   = detect_language(query)
    chunks = retrieve_chunks(
        query, faiss_index, chunk_map, embed_model,
        top_k=top_k, threshold=threshold
    )
    prompt = build_prompt(query, chunks)
    answer, elapsed = ask_llm(prompt)

    result = {
        "query"    : query,
        "lang"     : lang,
        "chunks"   : chunks,
        "prompt"   : prompt,
        "answer"   : answer,
        "time_s"   : elapsed,
    }

    if verbose:
        print(f"╔{'═'*60}")
        print(f"║ Query [{lang}]: {query}")
        print(f"║ Retrieved {len(chunks)} chunks | Time: {elapsed:.1f}s")
        print(f"╠{'─'*60}")
        print(f"║ ANSWER:")
        for line in answer.split("\n"):
            print(f"║  {line}")
        print(f"╚{'═'*60}\n")

    return result


In [5]:
# ════════════════════════════════════════════════════════════
# INCREMENTAL TEST — English Queries
# ════════════════════════════════════════════════════════════
# Test the full pipeline with English questions before multilingual.

english_tests = [
    "Who is the cardiologist and when is her clinic?",
    "What is the emergency ambulance number?",
    "What time does the pharmacy open?",
    "How do I get my blood test results?",
    "Can children visit ICU patients?",
]

print("🧪 TESTING ENGLISH QUERIES\n")
for q in english_tests:
    result = rag_answer(q, verbose=True)


🧪 TESTING ENGLISH QUERIES



NameError: name 'rag_answer' is not defined

In [ ]:
# ════════════════════════════════════════════════════════════
# INCREMENTAL TEST — Sinhala Queries
# ════════════════════════════════════════════════════════════

sinhala_tests = [
    "හදිසි ගිලන් රථ අංකය කුමක්ද?",    # What is the emergency number?
    "ළමා රෝග ශාඛාව කොතනද?",            # Where is paediatrics?
    "හමුවීමක් ලබා ගන්නේ කෙසේද?",       # How to book an appointment?
]

print("🧪 TESTING SINHALA QUERIES\n")
for q in sinhala_tests:
    result = rag_answer(q, verbose=True)


In [ ]:
# ════════════════════════════════════════════════════════════
# INCREMENTAL TEST — Tamil Queries
# ════════════════════════════════════════════════════════════

tamil_tests = [
    "அவசர சேவை எண் என்ன?",            # What is the emergency number?
    "தோல் நோய் மருத்துவர் யார்?",     # Who is the dermatologist?
    "மருந்தகம் எந்த நேரம் திறக்கும்?", # What time does pharmacy open?
]

print("🧪 TESTING TAMIL QUERIES\n")
for q in tamil_tests:
    result = rag_answer(q, verbose=True)


---
## 📊 Section 9 — Retrieval Quality Evaluation

**How do we know retrieval is working well?**  
We run a set of known Q&A pairs and check:
- **Hit@K:** Does the correct chunk appear in the top K results?
- **MRR (Mean Reciprocal Rank):** On average, how high does the right answer rank?
- **Average score:** Are our similarity scores strong?

This evaluation runs without needing Ollama — it only tests the retrieval stage.


In [ ]:
# ════════════════════════════════════════════════════════════
# RETRIEVAL EVALUATION
# ════════════════════════════════════════════════════════════

eval_pairs = [
    # (query, keyword_that_should_appear_in_top_result)
    ("Who is the cardiologist?",               "Cardiology"),
    ("What time does the pharmacy open?",      "Pharmacy"),
    ("Emergency ambulance number",             "1990"),
    ("How to book an appointment?",            "appointment"),
    ("Visiting hours for ICU",                 "ICU"),
    ("Which floor is orthopaedics?",           "Orthopaedics"),
    ("Who treats children?",                   "Paediatrics"),
    # Sinhala
    ("හදිසි ඒකකය කොතනද?",                   "Emergency"),
    # Tamil
    ("அவசர சேவை எண் என்ன?",                  "1990"),
]

hits, reciprocal_ranks, scores_list = [], [], []

for query, keyword in eval_pairs:
    results = retrieve_chunks(query, faiss_index, chunk_map, embed_model, top_k=5)
    hit = False
    rr  = 0.0
    for rank, r in enumerate(results, 1):
        if keyword.lower() in r["text"].lower():
            hit = True
            rr  = 1.0 / rank
            break
    hits.append(int(hit))
    reciprocal_ranks.append(rr)
    top_score = results[0]["score"] if results else 0.0
    scores_list.append(top_score)

    status = "✅" if hit else "❌"
    print(f"{status} [{detect_language(query)}] '{query[:50]}' | "
          f"RR={rr:.2f} | score={top_score:.3f}")

print(f"\n{'─'*50}")
print(f"Hit@5 : {sum(hits)/len(hits)*100:.1f}%")
print(f"MRR   : {sum(reciprocal_ranks)/len(reciprocal_ranks):.3f}")
print(f"Avg top score: {sum(scores_list)/len(scores_list):.3f}")


---
## 🌐 Section 10 — Write `app.py` & Launch Streamlit

**What happens here (two sub-steps):**

**Step 10-A — Write the Streamlit app to disk**  
A dedicated code cell writes the full `app.py` file using Python's `%%writefile` magic.  
This means `app.py` is **self-contained** — it imports nothing from the notebook.  
You can inspect, edit, or share it independently.

**Step 10-B — Launch Streamlit**  
A second cell runs `streamlit run app.py` as a subprocess.  
Open **http://localhost:8501** in your browser.

**Key design decisions in `app.py`:**
- `@st.cache_resource` — heavy models (embeddings + FAISS) load **once** per session
- Language is auto-detected from the Unicode script of each question
- The LLM is instructed to **answer in the same language as the question**  
  with an explicit prompt line: *"You MUST reply in {language}. Do NOT switch languages."*
- Only **1 best answer** is shown — no bullet dumps, no multiple candidates
- Score threshold `0.30` keeps only highly relevant chunks to avoid hallucination


In [7]:
%%writefile app.py

import streamlit as st
import sqlite3, os, time, json
from pathlib import Path

# ══════════════════════════════════════════════════════════════════
# PIPELINE — loaded once, cached for the whole session
# ══════════════════════════════════════════════════════════════════

@st.cache_resource(show_spinner=False)
def load_pipeline():
    """
    Build every heavy component ONCE and reuse across all queries.
    Streamlit reruns this file on every click; cache_resource prevents
    reloading the 120 MB embedding model each time.
    """
    from sentence_transformers import SentenceTransformer
    import faiss, numpy as np
    import ollama as _ollama

    embed_model = SentenceTransformer(
        "intfloat/multilingual-e5-base",
        device="cpu",
    )

    db_path = Path(__file__).parent / "hospital.db"
    conn    = sqlite3.connect(str(db_path), check_same_thread=False)
    _seed_db(conn)

    idx, chunk_map = _build_index(conn, embed_model)
    return embed_model, conn, idx, chunk_map, _ollama


# ── Database ──────────────────────────────────────────────────────
def _seed_db(conn):
    cur = conn.cursor()

    cur.execute("""CREATE TABLE IF NOT EXISTS doctors(
        id INTEGER PRIMARY KEY, name TEXT, dept TEXT, lang TEXT, bio TEXT)""")
    cur.executemany("INSERT OR IGNORE INTO doctors VALUES(?,?,?,?,?)", [
        (1,"Dr. Amara Perera","Cardiology","en",
         "Dr. Amara Perera is a senior cardiologist with 20 years of experience. "
         "She specialises in interventional cardiology and heart failure management. "
         "Clinic: Monday, Wednesday, Friday 9am-1pm. Phone: 0112-345-001. Floor 3."),
        (2,"Dr. Suresh Kumar","Neurology","en",
         "Dr. Suresh Kumar leads the Neurology department. "
         "He treats epilepsy, stroke, Parkinson's disease, and migraines. "
         "Clinic: Tuesday and Thursday 10am-2pm. Phone: 0112-345-002. Floor 3."),
        (3,"Dr. Nimali Fernando","Paediatrics","en",
         "Dr. Nimali Fernando is a paediatrician caring for children from birth to 18. "
         "Vaccines, growth monitoring, asthma, and childhood illnesses. "
         "Appointments: weekdays 8am-12pm. Floor 2. Phone: 0112-345-003."),
        (4,"Dr. Ravi Jayasinghe","Orthopaedics","en",
         "Dr. Ravi Jayasinghe specialises in joint replacement and sports injuries. "
         "Knee and hip replacement, shoulder surgery. "
         "Wednesday and Saturday 9am-3pm. Floor 4. Phone: 0112-345-004."),
        (5,"Dr. Priya Balasingham","Gynaecology","en",
         "Dr. Priya Balasingham provides maternal and women health services. "
         "Antenatal care, family planning, menopause management. "
         "Mon-Fri 9am-5pm. Floor 2. Phone: 0112-345-005."),
        (6,"Dr. Chamara Wijeratne","Oncology","en",
         "Dr. Chamara Wijeratne is head of Oncology. "
         "Treats breast cancer, lung cancer, and blood cancers. "
         "Tuesdays and Thursdays 9am-1pm. Floor 5. Phone: 0112-345-006."),
        (7,"Dr. Lakmini Dias","Endocrinology","en",
         "Dr. Lakmini Dias manages diabetes, thyroid disorders, and hormonal conditions. "
         "Clinic: Mondays and Fridays 10am-2pm. Floor 3. Phone: 0112-345-007."),
        (8,"Dr. Kamala Dissanayake","General Medicine","si",
         "Dr. Kamala Dissanayake සාමාන්‍ය වෛද්‍ය විශේෂඥ. "
         "ඇය දියවැඩියාව, රුධිර පීඩනය, උණ සහ සාමාන්‍ය රෝග සඳහා ප්‍රතිකාර කරයි. "
         "වේලාව: සඳුදා සිට සිකුරාදා 8am-4pm. 1 වන මහල. දු.අ: 0112-345-008."),
        (9,"Dr. Vijay Arulrajah","Dermatology","ta",
         "Dr. Vijay Arulrajah தோல் நோய் நிபுணர். "
         "சொரியாஸிஸ், அரிப்பு, முகப்பரு மற்றும் தோல் புற்றுநோய் சிகிச்சை வழங்குகிறார். "
         "நேரம்: திங்கள், புதன், வெள்ளி 9am-1pm. இரண்டாவது மாடி. தொலைபேசி: 0112-345-009."),
        (10,"Dr. Ananthi Sivakumar","Cardiology","ta",
         "Dr. Ananthi Sivakumar இதய நோய் நிபுணர். "
         "மாரடைப்பு, இதய செயலிழப்பு மற்றும் உயர் இரத்த அழுத்தம் சிகிச்சை செய்கிறார். "
         "கிளினிக்: செவ்வாய், வியாழன் 9am-1pm. மூன்றாவது மாடி. தொலைபேசி: 0112-345-010."),
    ])

    cur.execute("""CREATE TABLE IF NOT EXISTS departments(
        id INTEGER PRIMARY KEY, name TEXT, floor INTEGER, phone TEXT, hours TEXT)""")
    cur.executemany("INSERT OR IGNORE INTO departments VALUES(?,?,?,?,?)", [
        (1,"Cardiology",3,"0112-345-001","Mon-Fri 8am-5pm"),
        (2,"Neurology",3,"0112-345-002","Mon-Fri 9am-4pm"),
        (3,"Paediatrics",2,"0112-345-003","Mon-Sat 8am-6pm"),
        (4,"Orthopaedics",4,"0112-345-004","Mon-Fri 8am-5pm"),
        (5,"Gynaecology",2,"0112-345-005","Mon-Fri 9am-5pm"),
        (6,"Oncology",5,"0112-345-006","Tue-Thu 9am-1pm"),
        (7,"Endocrinology",3,"0112-345-007","Mon, Fri 10am-2pm"),
        (8,"General Medicine",1,"0112-345-008","Mon-Sat 7am-7pm"),
        (9,"Dermatology",2,"0112-345-009","Mon-Fri 9am-4pm"),
        (10,"Emergency",0,"0112-345-911","24 hours / 7 days"),
        (11,"Pharmacy",1,"0112-345-010","Mon-Sun 7am-10pm"),
        (12,"Radiology",1,"0112-345-011","Mon-Fri 8am-8pm"),
        (13,"Pathology Lab",1,"0112-345-012","Mon-Sat 6am-6pm"),
        (14,"ICU",4,"0112-345-013","24 hours — no direct calls"),
    ])

    cur.execute("""CREATE TABLE IF NOT EXISTS services(
        id INTEGER PRIMARY KEY, title TEXT, lang TEXT, detail TEXT)""")
    cur.executemany("INSERT OR IGNORE INTO services VALUES(?,?,?,?)", [
        (1,"Appointment Booking","en",
         "To book an appointment call 0112-345-000 or visit reception on Floor 1. "
         "Online booking: hospital.lk/appointments. Bring your National ID. "
         "Cancellations must be made 24 hours in advance."),
        (2,"Emergency Services","en",
         "24-hour Emergency Department is on the Ground Floor. Ambulance: 1990. "
         "The ER handles trauma, chest pain, stroke, breathing difficulty, unconsciousness, "
         "and all life-threatening conditions. Triage within 5 minutes of arrival."),
        (3,"Lab Tests","en",
         "Pathology Lab is on Floor 1, open Mon-Sat 6am-6pm. "
         "Fasting blood tests must be done before 10am. "
         "Results online within 24 hours at hospital.lk/results. "
         "Urine, stool, and culture tests available same day."),
        (4,"Visiting Hours","en",
         "General wards: 4pm-6pm weekdays; 10am-12pm and 4pm-6pm weekends. "
         "ICU: 10am-11am and 4pm-5pm daily, maximum 2 visitors per patient. "
         "Children under 12 not permitted in ICU or surgical wards."),
        (5,"Insurance and Billing","en",
         "We accept all major Sri Lankan health insurance. "
         "Cashless treatment: Ceylinco, AIA, Softlogic, Union Assurance. "
         "Billing office: Floor 1, Mon-Fri 8am-5pm. Phone: 0112-345-099."),
        (6,"Pharmacy","en",
         "Hospital pharmacy is on Floor 1, open Mon-Sun 7am-10pm. "
         "Prescriptions filled within 20 minutes. Generic medicines at reduced cost. "
         "Phone: 0112-345-010."),
        (7,"Radiology and Imaging","en",
         "X-ray, CT scan, MRI, and ultrasound available on Floor 1. "
         "CT and MRI require a doctor referral. "
         "X-ray results in 1 hour. MRI reports in 24 hours. Phone: 0112-345-011."),
        (8,"Parking","en",
         "Free parking for up to 2 hours for patients and visitors. "
         "Enter from Hospital Road South Gate. Paid parking: Rs. 100/hour thereafter. "
         "Disability bays at main entrance."),
        (9,"Patient Portal","en",
         "Register at hospital.lk/portal to view test results, book appointments, "
         "and download discharge summaries. "
         "Requires a valid email and mobile number used at registration."),
        # ── Sinhala ──────────────────────────────────────────────
        (10,"හදිසි සේවා","si",
         "හදිසි ගිලන් රථ අංකය: 1990. "
         "හෘදයාබාධ, ආඝාත රෝග ලක්ෂණ, දැඩි ලේ ගැලීම ඇතිවිට "
         "වහාම ගිලන් රථ ඇමතිය යුතුය. "
         "හදිසි ඒකකය 24 පැය, සතිය පුරා විවෘතව ඇත. ශූන්‍ය මහල."),
        (11,"හමුවීම් ලබා ගැනීම","si",
         "හමුවීමක් ලබා ගැනීම සඳහා 0112-345-000 අංකය අමතන්න "
         "හෝ 1 වන මහලේ reception ට ගොඩ ගන්න. "
         "ජාතික හැඳුනුම්පත රැගෙන එන්න. "
         "අන්තර්ජාල හමුවීම්: hospital.lk/appointments."),
        (12,"ළමා රෝග ශාඛාව","si",
         "ළමා රෝග ශාඛාව 2 වන මහලෙහි ඇත. "
         "Dr. Nimali Fernando ළමා රෝග විශේෂඥ. "
         "සඳුදා සිට සිකුරාදා 8am-12pm. "
         "දුරකථනය: 0112-345-003."),
        (13,"රසායනාගාර පරීක්ෂණ","si",
         "රසායනාගාරය 1 වන මහලෙහි ඇත. "
         "සඳුදා සිට සෙනසුරාදා 6am-6pm. "
         "නිරාහාර රුධිර පරීක්ෂණ 10am ට පෙර ගත යුතුය. "
         "ප්‍රතිඵල: hospital.lk/results. දුරකථනය: 0112-345-012."),
        (14,"ෆාමසිය","si",
         "ෆාමසිය 1 වන මහලෙහි ඇත. "
         "සඳුදා සිට ඉරිදා 7am-10pm. "
         "ප්‍රේෂකණ (prescriptions) විනාඩි 20 ක් ඇතුළත ලබා දෙයි. "
         "දුරකථනය: 0112-345-010."),
        # ── Tamil ────────────────────────────────────────────────
        (15,"அவசர சேவைகள்","ta",
         "அவசர ஆம்புலன்ஸ்: 1990. "
         "மாரடைப்பு, பக்கவாதம், அதிரடி காயம், உணர்விழப்பு ஆகியவற்றிற்கு "
         "உடனடியாக அழைக்கவும். "
         "அவசர பிரிவு 24 மணி நேரமும், 7 நாட்களும் திறந்திருக்கும். தரை மாடி."),
        (16,"சந்திப்பு பதிவு","ta",
         "சந்திப்பு பதிவு செய்ய 0112-345-000 என்ற எண்ணில் அழைக்கவும் "
         "அல்லது முதல் மாடியில் உள்ள வரவேற்பு மையத்தை சந்திக்கவும். "
         "தேசிய அடையாள அட்டை கொண்டு வரவும். "
         "ஆன்லைன் பதிவு: hospital.lk/appointments."),
        (17,"மருந்தகம்","ta",
         "மருந்தகம் முதல் மாடியில் உள்ளது. "
         "திங்கள் முதல் ஞாயிறு வரை 7am-10pm. "
         "மருத்துவர் பரிந்துரை மருந்துகள் 20 நிமிடத்தில் வழங்கப்படும். "
         "தொலைபேசி: 0112-345-010."),
        (18,"ஆய்வக பரிசோதனைகள்","ta",
         "பரிசோதனை ஆய்வகம் முதல் மாடியில் உள்ளது. "
         "திங்கள் முதல் சனிக்கிழமை 6am-6pm. "
         "உண்ணாவிரத இரத்த பரிசோதனைகள் காலை 10 மணிக்கு முன். "
         "முடிவுகள்: hospital.lk/results. தொலைபேசி: 0112-345-012."),
        (19,"தோல் நோய் சிகிச்சை","ta",
         "Dr. Vijay Arulrajah தோல் நோய் நிபுணர், இரண்டாவது மாடி. "
         "சொரியாஸிஸ், அரிப்பு, முகப்பரு, தோல் புற்றுநோய் சிகிச்சை. "
         "திங்கள், புதன், வெள்ளி 9am-1pm. தொலைபேசி: 0112-345-009."),
    ])
    conn.commit()


# ── FAISS Index ───────────────────────────────────────────────────
def _build_index(conn, embed_model):
    import faiss, numpy as np

    rows = []
    cur = conn.cursor()
    for doc_id, name, dept, lang, bio in cur.execute(
            "SELECT id, name, dept, lang, bio FROM doctors"):
        rows.append({"source":"doctors","lang":lang,
                     "text": f"Doctor Profile — {name} ({dept}): {bio}"})

    for d_id, name, floor, phone, hours in cur.execute(
            "SELECT id, name, floor, phone, hours FROM departments"):
        rows.append({"source":"departments","lang":"en",
                     "text": f"Department {name}: Floor {floor}, Phone {phone}, Hours {hours}."})

    for s_id, title, lang, detail in cur.execute(
            "SELECT id, title, lang, detail FROM services"):
        rows.append({"source":"services","lang":lang,
                     "text": f"{title}: {detail}"})

    texts = [r["text"] for r in rows]
    vecs  = embed_model.encode(
        texts, batch_size=16,
        show_progress_bar=False,
        normalize_embeddings=True,
    ).astype("float32")

    dim = vecs.shape[1]
    idx = faiss.IndexFlatIP(dim)
    idx.add(vecs)
    return idx, rows


# ── Language detection ────────────────────────────────────────────
def _detect_lang(text: str) -> str:
    """Detect script by Unicode code-point range. No ML needed."""
    for ch in text:
        cp = ord(ch)
        if 0x0D80 <= cp <= 0x0DFF: return "si"   # Sinhala
        if 0x0B80 <= cp <= 0x0BFF: return "ta"   # Tamil
    return "en"

_LANG_NAMES = {"en": "English", "si": "Sinhala", "ta": "Tamil"}


# ── Retrieval ─────────────────────────────────────────────────────
def _retrieve(query, embed_model, faiss_idx, chunk_map,
              top_k=6, threshold=0.30, lang_boost=0.05):
    import numpy as np

    user_lang = _detect_lang(query)
    q_vec = embed_model.encode(
        [query], normalize_embeddings=True
    ).astype("float32")

    scores, indices = faiss_idx.search(q_vec, top_k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx < 0 or idx >= len(chunk_map):
            continue
        chunk = chunk_map[idx].copy()
        adj   = float(score)
        if chunk.get("lang") == user_lang:
            adj += lang_boost          # reward same-language chunks
        if adj >= threshold:
            chunk["score"] = adj
            results.append(chunk)

    results.sort(key=lambda x: x["score"], reverse=True)
    return results, user_lang


# ── Prompt builder — strict single-language, single-answer ────────
def _build_prompt(query: str, chunks: list, lang: str) -> str:
    """
    Forces the LLM to:
      1. Answer in exactly the same language as the question.
      2. Give ONE concise answer — not multiple options.
      3. Use only the provided context — no hallucination.
    """
    lang_name = _LANG_NAMES.get(lang, "English")

    lang_rules = {
        "en": "Reply in clear English. Be concise.",
        "si": (
            "ඔබ පිළිතුර සිංහලෙන් පමණක් ලබා දෙන්න. "
            "ඉංග්‍රීසියට මාරු නොවන්න. "
            "නිවැරදි තොරතුරු පමණක් ලබා දෙන්න."
            "අලුත් තොරතුරු සකසන්න එපා."
            "වෛද්‍ය නාම (medical terms) ඉංග්‍රීසියෙන් ලිවිය හැකිය. "
            "කෙටිව හා පැහැදිලිව ලිවිය යුතුය."
        ),
        "ta": (
            "நீங்கள் தமிழில் மட்டுமே பதில் அளிக்க வேண்டும். "
            "ஆங்கிலத்திற்கு மாறாதீர்கள். "
            "மருத்துவ சொற்கள் ஆங்கிலத்தில் இருக்கலாம். "
            "சுருக்கமாக பதில் அளிக்கவும்."
        ),
    }

    context_block = " \n\n ".join(
        f"[{i+1}] {c['text']}" for i, c in enumerate(chunks)
    )

    return f"""You are a hospital information assistant.

STRICT RULES — follow all of them:
1. You MUST reply ONLY in {lang_name}. Do NOT switch to any other language.
2. Give exactly ONE direct answer. Do not list multiple options.
3. Use ONLY the information in the CONTEXT below. Do not guess or invent facts.
4. If the answer is not in the context, say (in {lang_name}):
   "I don't have that information. Please call reception: 0112-345-000."
5. {lang_rules.get(lang, lang_rules["en"])}

--- CONTEXT ---
{context_block}
--- END CONTEXT ---

Question: {query}

Answer (in {lang_name} only):"""


# ── LLM call ──────────────────────────────────────────────────────
def _ask_ollama(prompt: str, ollama, model: str) -> tuple:
    t0 = time.time()
    try:
        resp = ollama.chat(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            options={
                "temperature" : 0.1,   # near-zero = deterministic, factual
                "num_predict" : 400,   # cap tokens → faster on CPU
                "num_ctx"     : 2048,
                "num_thread"  : 8,
            },
        )
        return resp["message"]["content"].strip(), time.time() - t0, None
    except Exception as e:
        return None, time.time() - t0, str(e)


# ══════════════════════════════════════════════════════════════════
# STREAMLIT UI
# ══════════════════════════════════════════════════════════════════

def main():
    st.set_page_config(
        page_title="Hospital AI | රෝහල් AI | மருத்துவமனை AI",
        page_icon="🏥",
        layout="wide",
    )

    # ── Sidebar ──────────────────────────────────────────────────
    with st.sidebar:
        st.markdown("## ⚙️ Settings")
        model = st.selectbox(
            "Ollama Model",
            ["llama3.2:1b", "llama3.2", "gemma3:12b", "qwen3:4b", "phi3"],
            help="llama3.2:1b is fastest on CPU (~800 MB).",
        )
        top_k = st.slider("Context chunks retrieved", 2, 8, 5)
        st.divider()
        st.markdown("**Languages**")
        st.markdown("🇬🇧 English &nbsp; 🇱🇰 සිංහල &nbsp; 🇮🇳 தமிழ்",
                    unsafe_allow_html=True)
        st.divider()
        st.markdown("**Stack**")
        st.markdown("""
        🦙 Ollama (local LLM)

        🔍 FAISS vector search

        🗄️ SQLite knowledge base
        """)
        if st.button("🗑️ Clear chat"):
            st.session_state.history = []
            st.rerun()

    # ── Header ───────────────────────────────────────────────────
    st.markdown(
        "<h1 style='text-align:center;color:#1a6fa1;'>"
        "🏥 Hospital Multilingual AI Assistant</h1>"
        "<p style='text-align:center;color:#666;'>"
        "Ask in <b>English</b>, <b>සිංහල</b>, or <b>தமிழ்</b> — "
        "answered in the <b>same language</b> using local AI</p><hr/>",
        unsafe_allow_html=True,
    )

    # ── Load pipeline ─────────────────────────────────────────────
    with st.spinner("Loading AI models… (first run ~30 s)"):
        try:
            embed_model, conn, faiss_idx, chunk_map, ollama = load_pipeline()
        except Exception as e:
            st.error(f"Failed to load pipeline: {e}")
            st.stop()
    st.success("✅ Ready! Type a question below.", icon="🚀")

    # ── Sample questions ──────────────────────────────────────────
    st.markdown("#### 💡 Sample questions — click to try:")
    cols = st.columns(3)
    samples = [
        ("🇬🇧 English",  ["Who is the cardiologist?",
                          "What time does the pharmacy close?",
                          "How do I book an appointment?"]),
        ("🇱🇰 සිංහල",  ["හදිසි ගිලන් රථ අංකය කුමක්ද?",
                          "ෆාමසිය කීයට වහනවාද?"]),
        ("🇮🇳 தமிழ்",  ["அவசர சேவை எண் என்ன?",
                          "மருந்தகம் எந்த நேரம் திறக்கும்?"]),
    ]
    chosen = None
    for col, (label, qs) in zip(cols, samples):
        with col:
            st.markdown(f"**{label}**")
            for q in qs:
                if st.button(q, key=q, use_container_width=True):
                    chosen = q

    st.divider()

    # ── Chat history ──────────────────────────────────────────────
    if "history" not in st.session_state:
        st.session_state.history = []   # list of (role, text)

    for role, msg in st.session_state.history:
        with st.chat_message(role):
            st.markdown(msg)

    # Accept typed or button-clicked query
    typed = st.chat_input(
        "Ask your question… / ඔබගේ ප්‍රශ්නය… / உங்கள் கேள்வி…"
    )
    query = typed or chosen
    if not query:
        return

    # Show user bubble
    with st.chat_message("user"):
        st.markdown(query)
    st.session_state.history.append(("user", query))

    # ── Generate answer ───────────────────────────────────────────
    with st.chat_message("assistant"):
        lang = _detect_lang(query)
        lang_label = {"en":"🇬🇧 English","si":"🇱🇰 Sinhala","ta":"🇮🇳 Tamil"}.get(lang,"?")

        with st.spinner(f"Searching knowledge base ({lang_label})…"):
            chunks, _ = _retrieve(query, embed_model, faiss_idx, chunk_map, top_k)

        if not chunks:
            no_info = {
                "en": "I don't have that information. Please call reception: 0112-345-000.",
                "si": "ඒ තොරතුරු මා සතු නැත. කරුණාකර 0112-345-000 අමතන්න.",
                "ta": "அந்த தகவல் என்னிடம் இல்லை. 0112-345-000 என்ற எண்ணில் அழைக்கவும்.",
            }
            answer = no_info.get(lang, no_info["en"])
            st.markdown(answer)
        else:
            with st.expander(
                f"📄 {len(chunks)} relevant chunk(s) retrieved (click to inspect)"
            ):
                for i, c in enumerate(chunks, 1):
                    st.markdown(
                        f"**[{i}]** `{c['source']}` · score `{c['score']:.3f}`\n\n"  

                        f"{c['text'][:180]}{'…' if len(c['text'])>180 else ''}"
                    )

            prompt = _build_prompt(query, chunks, lang)

            with st.spinner("Generating answer with local LLM…"):
                t0 = time.time()
                answer, elapsed, err = _ask_ollama(prompt, ollama, model)

            if err:
                answer = (
                    f"⚠️ LLM error: {err} \n\n"


                    "Make sure Ollama is running: open a terminal and run `ollama serve`"
                )
                st.error(answer)
            else:
                st.markdown(answer)
                st.caption(
                    f"⏱ {elapsed:.1f}s · model: `{model}` · "
                    f"lang detected: `{lang}` · chunks used: {len(chunks)}"
                )

    st.session_state.history.append(("assistant", answer))

    # ── DB Explorer ───────────────────────────────────────────────
    with st.expander("🗄️ Hospital Database Explorer"):
        import pandas as pd
        t1, t2, t3 = st.tabs(["Doctors", "Departments", "Services"])
        cur = conn.cursor()
        with t1:
            st.dataframe(pd.read_sql(
                "SELECT id,name,dept,lang FROM doctors", conn),
                use_container_width=True)
        with t2:
            st.dataframe(pd.read_sql(
                "SELECT * FROM departments", conn),
                use_container_width=True)
        with t3:
            st.dataframe(pd.read_sql(
                "SELECT id,title,lang,detail FROM services", conn),
                use_container_width=True)


if __name__ == "__main__":
    main()


Overwriting app.py


In [ ]:
!streamlit run app.py